## Lesson 02 Embeddings
How to Represent, Encode, and Compare Text Using Vector Spaces

In [1]:
"""
======================================================================
Model Selection: Sentence Transformers
======================================================================
We use all-MiniLM-L6-v2, a small, fast model suitable for short English text (e.g., FAQs).
- 384-dim embeddings (compact)
- Fast on CPU
- Good general-purpose quality
- Uses cosine similarity for comparisons

The model and tokenizer (~80 MB) are downloaded on first run and cached locally.
"""
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
"""
======================================================================
Embedding Example
======================================================================
model is an instance of the SentenceTransformer class. model.encode()
is a method on that instance; it takes a string, runs it through the
neural network, and returns a fixed-length numeric array called an embedding.

For all-MiniLM-L6-v2 that array is always 384 numbers. Each number is a
value along one of 384 learned dimensions of meaning. These dimensions are
not labelled; the model learned them implicitly during training and they
don't map to anything human-readable like "topic" or "sentiment."

v1.shape returns (384,); a 1D array of 384 values. The trailing comma is
Python tuple notation, the same as DataFrame.shape. Encoding
a batch of N sentences at once would return shape (N, 384) instead.

We print the first 5 values as a preview to confirm the array exists.
"""
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

print(f"Query: {q1}")
print(f"Embedding shape: {v1.shape}")
print(f"Embedding (first 5 values): {v1[:5]}")

Query: Can I still join the course after the start date?
Embedding shape: (384,)
Embedding (first 5 values): [ 0.02139038 -0.07398     0.00142069  0.02138165  0.02451134]


In [3]:
"""
======================================================================
Encode Document
======================================================================
In RAG, "document" refers to any chunk of text in your index; here it's
a short FAQ answer. We encode it the same way as the query, so both live
in the same vector space and can be compared by similarity.
"""
d = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

print(f"Document: {d}")
print(f"Embedding shape: {dv.shape}")
print(f"Embedding (first 5 values): {dv[:5]}")

Document: You don't need to register. You're accepted. You can also just start learning and submitting homework without registering.
Embedding shape: (384,)
Embedding (first 5 values): [-0.0481207  -0.12994207 -0.00486808  0.01374956 -0.00851095]


In [4]:
"""
======================================================================
Compare Query and Document with Dot Product
======================================================================
Compute the dot product between the query embedding (v1) and document
embedding (dv). Because all-MiniLM-L6-v2 outputs normalised vectors,
the dot product equals cosine similarity.

The query asks about joining the course late. The document answers a
registration question; related, but not a direct answer. A score of
~0.32 reflects that loose topical overlap.

This interpretation only works because we can read both texts ourselves.
In practice, raw scores have no universal meaning; thresholds vary by
model, domain, and dataset. The score is most useful comparatively:
rank all documents and surface the highest ones. A single number only
earns meaning relative to the other scores in your index.
"""
similarity = v1.dot(dv)
print(f"Dot product similarity (v1 · dv): {similarity}")

Dot product similarity (v1 · dv): 0.3233240246772766


In [5]:
"""
======================================================================
Encode Unrelated Query
======================================================================
We encode a second query on a completely different topic; Docker
installation has nothing to do with course registration or enrollment.

In the next cell we compute v2.dot(dv) and compare it against v1.dot(dv)
(0.32). That contrast will show whether the model has captured the
difference in meaning between a relevant and an irrelevant query.
"""
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

print(f"Query: {q2}")
print(f"Embedding shape: {v2.shape}")
print(f"Embedding (first 5 values): {v2[:5]}")

Query: How to install Docker on Windows?
Embedding shape: (384,)
Embedding (first 5 values): [-0.02356781  0.07855558  0.02337921 -0.00672476 -0.02588748]


In [6]:
"""
======================================================================
Compare Unrelated Query with Document
======================================================================
v2.dot(dv) scores the Docker query against the registration document.
The result (0.02) against v1.dot(dv) (0.32) confirms the model has
captured the difference in meaning.

Neither number is meaningful in isolation; but together they are. The
gap between 0.32 and 0.02 is the signal. This is the core mechanic of
vector search: not a threshold, but a ranking. The model doesn't decide
"relevant or not" — it assigns scores, and the highest ones win.
"""
similarity_unrelated = v2.dot(dv)
print(f"Dot product similarity (v2 · dv): {similarity_unrelated}")

Dot product similarity (v2 · dv): 0.019730502739548683


## Lesson 03 Embedding Our Dataset
Embedding the Full FAQ Dataset: Batching and Vector Generation

In [7]:
"""
======================================================================
Load FAQ Dataset
======================================================================
Import and use the load_faq_data() function from ingest.py.
This loads the FAQ documents (each with a question and answer).
"""
from ingest import load_faq_data

documents = load_faq_data()
print(f"Loaded {len(documents)} FAQ documents")

Loaded 1350 FAQ documents


In [8]:
"""
======================================================================
Combine Question and Answer into Single Text
======================================================================
Each document in the FAQ dataset is a dictionary with separate "question"
and "answer" fields. Before we can embed a document, we need to represent
it as a single string.

We concatenate both fields so  the embedding captures meaning from the full
document; not just the question or just the answer. This matters because
a user query might match the wording in the answer rather than the question,
or vice versa. Keeping them together gives the embedding model the most
context to work with.

The result is a flat list of strings, one per document, ready to be
passed to model.encode() in batches. The original documents list is
unchanged; texts is just a parallel list of combined strings we use
for embedding.
"""
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

print(f"Built {len(texts)} combined text entries")
print(f"First example: {texts[0][:100]}...")

Built 1350 combined text entries
First example: Course: When does the course start? A new cohort runs roughly January–April every year. For the curr...


In [10]:
"""
======================================================================
Prepare Visual Progress Tracking (tqdm)
======================================================================
The next step involves passing our ~1,200 text entries through a machine 
learning model in batches of 50. Because model encoding is a heavy 
computation (especially on a CPU), it will take some time to finish.

Without a tracking mechanism, your screen would stay completely blank, 
making it look like the code is frozen or stuck.

To fix this, we import 'tqdm.auto'. We will wrap our batch loop in this 
utility so it can display a live, dynamic progress bar showing us 
exactly how many batches have been processed, the percentage completion, 
and the estimated time remaining.

Why 'tqdm.auto'?
It automatically adapts to your workspace:
  - Jupyter Notebooks: Shows a clean, colorful graphic widget bar.
  - Standard Terminals: Shows a smooth text-based character bar.
"""
from tqdm.auto import tqdm
print("tqdm.auto successfully imported and ready for batch tracking.")

tqdm.auto successfully imported and ready for batch tracking.


In [11]:
"""
======================================================================
Generate Embeddings Using Batching
======================================================================
Process texts in batches of 50 instead of all at once:
- Avoids long wait times without visibility
- Works well on CPU (Codespaces without GPU)

This is a one-off cost; embeddings are cached for later use.
"""

batch_size = 50
vectors = []

# Loop through the texts in steps of 50, using tqdm to display a live progress bar.
for i in tqdm(range(0, len(texts), batch_size)):
    
    # Slice the list from 'i' to 'i + batch_size' to extract the next chunk (e.g., indices 0 to 50, then 50 to 100).
    batch = texts[i:i + batch_size]
    
    # Pass the batch to the model to generate 50 vector embeddings.
    batch_vectors = model.encode(batch)
    
    # Add the new vectors to our main list. We use .extend() instead of .append() to keep the list flat, avoiding nested lists.
    vectors.extend(batch_vectors)

# Print the final vector count to confirm all texts were processed.
print(f"Generated {len(vectors)} embedding vectors")

  0%|          | 0/27 [00:00<?, ?it/s]

Generated 1350 embedding vectors


In [12]:
"""
======================================================================
Convert Vectors to 2D NumPy Array (Matrix)
======================================================================
Standard Python lists are too slow for the heavy math needed in the 
next step (similarity search). Converting the data into a NumPy array 
optimizes it for lightning-fast mathematical calculations.

Transform the list of vectors into a NumPy array where:
- Rows = individual FAQ documents (embedding vectors)
- Columns = embedding dimensions (384 from all-MiniLM-L6-v2)

"""
import numpy as np

X = np.array(vectors)
print(f"Matrix shape: {X.shape}")
print(f"Number of documents (embedding vectors): {X.shape[0]}")
print(f"Embedding dimensions: {X.shape[1]}")

Matrix shape: (1350, 384)
Number of documents (embedding vectors): 1350
Embedding dimensions: 384


## Lesson 04 Vector Search
Vector Search Under the Hood: Scoring, Best Match, and Top 5 Retrieval with NumPy

In [13]:
"""
======================================================================
Encode Query for Vector Search
======================================================================
Encode a query string into its embedding vector.
This vector will be compared against all document embeddings in matrix X in teh next step.
"""
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

print(f"Query: {query}")
print(f"Query embedding shape: {v_query.shape}")
print(f"Query embedding (first 5 values): {v_query[:5]}")

Query: Can I still join the course after the start date?
Query embedding shape: (384,)
Query embedding (first 5 values): [ 0.02139038 -0.07398     0.00142069  0.02138165  0.02451134]


In [14]:
"""
======================================================================
Compute Similarity Scores Using Matrix-Vector Multiplication
======================================================================
Compute dot product between query vector and all document embeddings.
X.dot(v_query) performs matrix-vector multiplication:
- Each element i = cosine similarity between document i and query
- Much faster than Python loops (optimized C code in NumPy)
- Result: one score per document (The maximum score = the most relevant document)
"""
scores = X.dot(v_query)

print(f"Number of scores: {len(scores)}")
print(f"Scores range: {scores.min():.3f} to {scores.max():.3f}")
print(f"First 5 scores: {scores[:5]}")

Number of scores: 1350
Scores range: -0.145 to 0.763
First 5 scores: [0.4874058  0.20991945 0.7629411  0.44378543 0.26084003]


In [15]:
"""
======================================================================
Compute Similarity Scores Using For Loop (Alternative)
======================================================================
This loop computes the exact same dot products as our fast matrix 
multiplication (X.dot), but does it sequentially (one by one). 

It is useful for understanding how the math works under the hood, but 
is significantly slower.

WHY DOES THE CHECK BELOW RETURN 'FALSE'?
When you run the print statement, it will likely output False. This is 
not a bug! It happens because of microscopic floating-point rounding 
differences.

The optimized matrix math (X.dot) groups and adds numbers differently 
than a strict Python loop. This causes tiny variations at the ~8th 
decimal place. Because Python is extremely strict by default, it flags 
them as unequal. We will fix this in the next cell!
"""
scores_loop = [v_query.dot(X[i]) for i in range(len(X))]

# This will likely print False due to microscopic rounding differences
print(f"Loop matches matrix version: {np.allclose(scores, scores_loop)}")

Loop matches matrix version: False


In [16]:
"""
======================================================================
Fixing the Precision Check (np.allclose & atol)
======================================================================
In the previous cell, we saw that microscopic rounding errors caused 
our mathematically identical arrays to be flagged as unequal (False).

HOW DO WE FIX THIS?
We use NumPy's `np.allclose()` function with a custom tolerance. 
Instead of asking Python "Are these exactly equal?", we ask "Are these 
close enough?"

- `atol` stands for Absolute Tolerance. 
- Setting `atol=1e-5` tells NumPy: "Consider these arrays a perfect match 
  as long as the difference between them is smaller than 0.00001."

For machine learning purposes, a difference of 0.00001 is irrelevant.
"""
# We don't need to recalculate the loop here if we just ran it above, 
# but here is the updated check:

# Check if they match, allowing for a microscopic 0.00001 margin of error
print(f"Loop matches matrix version: {np.allclose(scores, scores_loop, atol=1e-5)}")

Loop matches matrix version: True


In [17]:
"""
Find the Best Matching Document

At this point, scores already contains one similarity score per document.
Now we need to find which document got the highest score.

np.argmax(scores) returns the index of the largest value in the scores array.
That index tells us which document to retrieve from documents.

The print statement uses an f-string for display, and .3f means show the
score rounded to 3 decimal places.
"""
idx = np.argmax(scores)

print(f"Best match index: {idx}")
print(f"Best match score: {scores[idx]:.3f}")

Best match index: 2
Best match score: 0.763


In [18]:
"""
======================================================================
View the Best Matching Document
======================================================================
Retrieve and display the full FAQ document (question + answer)
for the best matching index.
"""
best_doc = documents[idx]

print(f"Best match document:")
print(f"  ID: {best_doc['id']}")
print(f"  Course: {best_doc['course']}")
print(f"  Section: {best_doc['section']}")
print(f"  Question: {best_doc['question']}")
print(f"  Answer: {best_doc['answer']}")

Best match document:
  ID: 3f1424af17
  Course: data-engineering-zoomcamp
  Section: General Course-Related Questions
  Question: Course: Can I still join the course after the start date?
  Answer: Yes, even if you don't register, you're still eligible to submit the homework.

Be aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.


In [19]:
"""
======================================================================
Get Top 5 Documents (Two-Step Method)
======================================================================
We usually want more than the single best match because the answer may be
spread across multiple documents.

The top result is not always perfect, and sometimes the second or third
result adds useful context. Returning several candidates gives the LLM a
better chance to combine the relevant pieces into a stronger answer.

np.argsort creates a sorted index order based on a temporary sorted view
of the scores:
- It returns document indices from lowest score to highest score
- We take the last 5 indices to get the highest scores
- We reverse them so they are ordered highest-first
"""
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

print(f"Top 5 indices: {top5}")
print(f"Top 5 scores: {scores[top5]}")

Top 5 indices: [  2 625 907 538   7]
Top 5 scores: [0.7629411  0.7579372  0.71921325 0.6536313  0.56009996]


In [20]:
"""
======================================================================
Get Top 5 Documents (One-Line Max-Sort Trick)
======================================================================
Use -scores inside argsort to sort in descending order:
- argsort creates a temporary negated view for sorting
- the temporary negative values make the largest score become the smallest
- argsort then returns the best matches first
- the printed scores below are still the original positive scores
"""
top5_short = np.argsort(-scores)[:5]

print(f"Top 5 indices (short method): {top5_short}")
print(f"Top 5 scores: {scores[top5_short]}")

Top 5 indices (short method): [  2 625 907 538   7]
Top 5 scores: [0.7629411  0.7579372  0.71921325 0.6536313  0.56009996]


In [21]:
"""
======================================================================
Display Top 5 Matching Documents
======================================================================
Loop through top 5 indices and print each document's score
and full content (question + answer).
"""
for idx in top5:
    print(f"Score: {scores[idx]:.3f}")
    print(f"Document: {documents[idx]}")
    print()  # Blank line between documents

Score: 0.763
Document: {'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

Score: 0.758
Document: {'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

Score: 0.719
Document: {'id': '41aabbd7c5', 'course': 'machine-learning-zoomca

## Lesson 05 Vector Search with minsearch
How to Represent, Encode, and Compare Text Using Vector Spaces
In-memory vector search for quick experiments and RAG integration

In [22]:
"""
======================================================================
Vector Search with minsearch
======================================================================
Use minsearch to store document embeddings in memory and search them later:
- fit builds the in-memory index from embeddings and documents
- search compares a query vector against stored vectors
- filter_dict can restrict matches by keyword fields like course
- this is a simple first step before using a persistent vector database
"""
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

print(f"Indexed {len(documents)} documents with {X.shape[1]}-dimensional vectors")
print("We pass the NumPy array X with all embeddings and the list of documents as payload.")
print("The keyword_fields parameter works the same as in the text Index, so we can filter by course later.")

Indexed 1350 documents with 384-dimensional vectors
We pass the NumPy array X with all embeddings and the list of documents as payload.
The keyword_fields parameter works the same as in the text Index, so we can filter by course later.


In [23]:
"""
======================================================================
Searching
======================================================================
Search the vector index with a query embedding:
- encode the query into a vector first
- search compares the query vector against all stored document vectors
- minsearch returns results in best-first order
- results[0] is the top-scoring match
- in this example, that should be the late-join course FAQ with metadata
"""
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)
top_result = results[0]

print(f"Query: {query}")
print(f"Query vector shape: {query_vector.shape}")
print(f"Number of results: {len(results)}")
print(f"Top Result:")
top_result

Query: I just discovered the course. Can I still join it?
Query vector shape: (384,)
Number of results: 5
Top Result:


{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [24]:
"""
======================================================================
Filtering by Course
======================================================================
Filter vector search by course before ranking:
- keyword filters narrow the candidate set first
- this improves relevance for users in a specific course
- only documents from the chosen course are scored
- after filtering, the query still uses vector similarity for ranking
"""
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

print(f"Number of filtered results: {len(results)}")
print(f"Top filtered results:")
results[0:5]

Number of filtered results: 5
Top filtered results:


[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

### Lesson 06 — RAG with Vector Search
Replacing keyword search with semantic search in the RAG pipeline

In [33]:
"""
======================================================================
RAG with Vector Search
======================================================================
Replace keyword search with vector search inside the RAG pipeline:
- the overall RAG flow stays the same
- only the search step changes
- build_prompt and llm remain unchanged
- this keeps the pipeline modular and easy to swap from keyword search to vector search
"""
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

print("RAG function defined successfully.")

RAG function defined successfully.


In [34]:
"""
======================================================================
Using RAGBase
======================================================================
Set up the RAG helper and an OpenAI-compatible client for Gemini:
- RAGBase keeps the RAG flow modular
- search handles retrieval
- build_prompt formats the context
- llm sends the prompt to the model
- the client is configured to call Gemini through the OpenAI interface
"""
import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI(
    # Fetch the loaded Gemini key
    api_key=os.getenv("GEMINI_API_KEY"),

    # Reroute standard OpenAI traffic to Google's servers
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

print("Gemini-compatible OpenAI client ready.")

Gemini-compatible OpenAI client ready.


In [37]:
"""
======================================================================
Load Data and Build Index
======================================================================
Load the FAQ data and build the text-search index:
- documents holds the FAQ entries
- build_index creates the searchable index
- this is the same setup used in module 1
"""
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

print(f"Loaded {len(documents)} documents.")
print("Text index built successfully.")

Loaded 1350 documents.
Text index built successfully.


In [38]:
"""
======================================================================
Test RAGBase
======================================================================
Create the base RAG assistant and test the keyword-search pipeline:
- RAGBase uses the index directly
- search still works on raw query text
- this is the baseline before swapping in vector search
"""
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

query = "I just found out about the program, can I still sign up?"
answer = assistant.rag(query)

print(f"Query: {query}")
print("RAGBase response generated.")
print(answer)

Query: I just found out about the program, can I still sign up?
RAGBase response generated.
Yes, you can still join, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [39]:
"""
======================================================================
Subclass for Vector Search
======================================================================
Use the same RAG structure as before, but change retrieval so that:
- the query is embedded first
- the vector index searches using that embedding
- the course filter is still applied
- everything else comes from RAGBase
"""
class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        # super() invokes RAGBase's __init__ so this child gets the object's starting attributes and values, despite having its own __init__.
        # **kwargs forwards the extra keyword arguments required by RAGBase.__init__, so the parent initializer can run properly.
        super().__init__(**kwargs)

        # Save the embedding model so we can turn query text into vectors later.
        self.embedder = embedder

    def search(self, query, num_results=5):
        # Convert the user's text query into a vector before searching.
        query_vector = self.embedder.encode(query)

        # Keep the search limited to the current course.
        filter_dict = {"course": self.course}

        # Search the vector index with the query vector instead of raw text.
        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

print("RAGVector class defined successfully.")

RAGVector class defined successfully.


In [40]:
"""
======================================================================
Use Vector Search in RAG
======================================================================
Create the vector-based assistant and run the same question again:
- only the search step has changed
- the prompt and LLM call remain the same
- vector search helps with rephrased questions
"""
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

vector_answer = vector_assistant.rag("I just found out about the program, can I still sign up?")

print("Vector RAG response generated.")
print(vector_answer)

Vector RAG response generated.
Yes, you can still join, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [45]:
"""
======================================================================
Using It
======================================================================
Create the vector-based assistant and try it on a rephrased question:
- the assistant uses vector search instead of keyword search
- rephrased questions can still match the right FAQ
- only the search step changed; the rest of the RAG pipeline stays the same
- the same modular design also lets us swap the LLM step later
"""

rephrased_question = "I missed the kickoff, am I still allowed to take part?"

vector_response = vector_assistant.rag(rephrased_question)

print("Vector RAG answer:")
print(response)

Vector RAG answer:
Yes, you can still join. If you want to receive a certificate, you just need to make sure you submit your project while submissions are still being accepted.


In [46]:
"""
======================================================================
Testing the Rephrased Question Using RAGBase
======================================================================
Create the base assistant and try it on the same rephrased question:
- this checks the non-vector retrieval path
- the same modular structure keeps the code easy to compare
"""
answer = assistant.rag(rephrased_question)

print("RAGBase answer:")
print(answer)

RAGBase answer:
I don't know.


### Lesson 07 Vector Search with sqlitesearch
Persistent vector search with SQLite and approximate nearest neighbors

In [97]:
"""
======================================================================
Vector Search with sqlitesearch
======================================================================
- sqlitesearch is the persistent, on-disk sibling of minsearch.
- It stores vectors in SQLite so the index can be reused across processes and restarts.
- It supports ANN retrieval, where the search first narrows to a candidate set and then reranks exactly.
- Available modes:
  * lsh  -> default, random hyperplane projections, good up to ~100K vectors
  * ivf  -> K-means clustering, good for ~10K-500K vectors
  * hnsw -> proximity graph, highest recall, good for ~10K-1M+ vectors
- For this small dataset, the notes say `lsh` is fine and I have used it below.
[HOWEVER, the course notes actually use `ivf`]
- All modes use two-phase search: approximate candidate retrieval first, then exact cosine similarity reranking.
"""

from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="lsh",
    db_path="faq_vectors2.db",
)

print(
    "Created sqlitesearch VectorSearchIndex with mode='lsh', "
    "keyword_fields=['course'], and db_path='faq_vectors2.db'."
)

Created sqlitesearch VectorSearchIndex with mode='lsh', keyword_fields=['course'], and db_path='faq_vectors2.db'.


In [71]:
"""
======================================================================
Indexing the data with sqlitesearch
======================================================================
- Fit the vector index with the embedding matrix and the original documents.
- The index is written to disk at faq_vectors2.db.
- Unlike minsearch, this index persists across restarts.
- You can search immediately after indexing, or reopen the same index later without rebuilding it.
"""

# Clear any existing contents first, since fit() only works on an empty index.
vs_index.clear()
vs_index.fit(vectors, documents)

print(f"Indexed {len(documents)} documents and saved the vector index to 'faq_vectors2.db'.")

Indexed 1350 documents and saved the vector index to 'faq_vectors2.db'.


In [72]:
"""
======================================================================
Searching with sqlitesearch
======================================================================
- Search works like minsearch, but the query must be embedded first.
- This is one reason vector search is heavier than text search.
- Text search can take the raw query directly, but vector search needs a query vector in the same space as the indexed documents.
- This first search is unfiltered, so we can see the raw top matches before applying the course filter in the next cell.
"""

query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

print("Encoded the query into a vector and searched for the top 5 nearest matches without filtering.")
print(f"Query: {query}")
print(f"Number of results returned: {len(results)}")
print("This is the unfiltered search result before applying the course filter.")
results

Encoded the query into a vector and searched for the top 5 nearest matches without filtering.
Query: I just discovered the course. Can I still join it?
Number of results returned: 5
This is the unfiltered search result before applying the course filter.


[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [73]:
"""
======================================================================
Filtering by course with sqlitesearch
======================================================================
- Filtering works the same way as in minsearch.
- The query is still encoded into a vector first.
- The filter narrows results to the selected course before returning the top matches.
"""

results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

print("Applied a course filter for 'llm-zoomcamp' and searched the top 5 vector matches.")
print(f"Number of filtered results returned: {len(results)}")
print("The search was restricted to the selected course before ranking the results.")
results

Applied a course filter for 'llm-zoomcamp' and searched the top 5 vector matches.
Number of filtered results returned: 5
The search was restricted to the selected course before ranking the results.


[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '85384a18e5',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
  'answer': "No, you don't have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.\n\nSee the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md#openai-api-alternatives)."},
 {'id': 'dbf5369006'

In [74]:
"""
======================================================================
Closing the sqlitesearch index
======================================================================
- Close the connection when you're done using the index.
- This releases resources and finalizes the persistent index cleanly.
"""

vs_index.close()

print("Closed the sqlitesearch vector index connection.")

Closed the sqlitesearch vector index connection.


In [88]:
"""
======================================================================
Reopening the sqlitesearch index
======================================================================
- In a new Python session, you can reopen the existing index from disk.
- You still need the embedding model to encode the query.
- You do not need to re-embed the full dataset or call fit() again.
- This is the same ingestion/query split used for persistent text search.
- A similar but not identical query is used to show that the reopened
index is generating fresh search results, not just returning saved output
from the previous cell.
======================================================================
"""

from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex
from pathlib import Path

db_path = "faq_vectors2.db"

print("DB exists before reopen:", Path(db_path).exists())

model = SentenceTransformer("all-MiniLM-L6-v2")

# Reopening the connection using the corrected "lsh" mode
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="lsh",
    db_path=db_path
)

query = "Is it possible to enroll late?"
query_vector = model.encode(query)

results = vs_index.search(query_vector,filter_dict={"course": "llm-zoomcamp"}, num_results=5)

print(f"Query: {query}")
print(f"Number of results returned: {len(results)}")
results

DB exists before reopen: True


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query: Is it possible to enroll late?
Number of results returned: 3


[{'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form 

In [90]:
"""
======================================================================
Testing sqlitesearch index with a Non-relevant Question
======================================================================
- The out-of-domain query ("How do I run Kafka?").
"""

query = "How do I run Kafka?"
query_vector = model.encode(query)
results = vs_index.search(query_vector,filter_dict={"course": "llm-zoomcamp"}, num_results=5)

print("Queried the reopened sqlitesearch index without rebuilding embeddings.")
print(f"Query: {query}")
print(f"Number of results returned: {len(results)}")
results

Queried the reopened sqlitesearch index without rebuilding embeddings.
Query: How do I run Kafka?
Number of results returned: 3


[{'id': 'e5d8a2c761',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Project: what does "reproducibility" mean — do reviewers need access to my API keys?',
  'answer': "Never share API keys or hosted-service credentials in your repo. Reproducibility means a peer reviewer can clone the repo and follow your README to recreate the system from scratch — using their own credentials.\n\nConcretely:\n\n- Provide a script (or notebook) that ingests the dataset and (re)builds the search index locally.\n- Ship a `.env.example` with the variable names but no values; have the reviewer create their own `.env` with their own keys. Keep `.env` in `.gitignore`.\n- Use a cheap model (`gpt-4o-mini`, Groq, etc.) so reviewers don't burn through credits when running your project.\n- Pin dependency versions (`requirements.txt` / `pyproject.toml` lock file) and document the Python version (and Docker version, if used)."},
 {'id': '054f3fd58f',
  'course': 'llm-zoomcamp',
  'secti

In [91]:
"""
======================================================================
Testing sqlitesearch index without the filter
======================================================================
- The previously out-of-domain query ("How do I run Kafka?").
"""

query = "How do I run Kafka?"
query_vector = model.encode(query)
results = vs_index.search(query_vector, num_results=5)

print("Queried the reopened sqlitesearch index without rebuilding embeddings.")
print(f"Query: {query}")
print(f"Number of results returned: {len(results)}")
results

Queried the reopened sqlitesearch index without rebuilding embeddings.
Query: How do I run Kafka?
Number of results returned: 5


[{'id': '5ca6890c1a',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```'},
 {'id': 'cd8a62fc55',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BO

In [92]:
"""
======================================================================
Using sqlitesearch vector search in RAG
======================================================================
- Load the embedding model.
- Open the persistent vector index from disk.
- This is the setup step before plugging vector search into the RAG pipeline.
"""

from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="lsh",
    db_path="faq_vectors2.db"
)

print("Loaded the embedding model and reopened the persistent sqlitesearch index for RAG.")
print("The index is ready to be used for vector retrieval inside the RAG pipeline.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded the embedding model and reopened the persistent sqlitesearch index for RAG.
The index is ready to be used for vector retrieval inside the RAG pipeline.


In [93]:
"""
======================================================================
Using sqlitesearch vector search in RAG
======================================================================
- Reuse the persistent vector index inside the RAG pipeline.
- RAGVector overrides only the search step.
- The query is embedded first, then passed into the vector index.
- Everything else stays inherited from RAGBase.
"""

import os
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    # Fetch the loaded Gemini key
    api_key=os.getenv("GEMINI_API_KEY"),

    # Reroute standard OpenAI traffic to Google's servers
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

print("Created RAGVector with the persistent sqlitesearch index and the OpenAI client.")
print("The search method now embeds the query before retrieving the top matches.")

Created RAGVector with the persistent sqlitesearch index and the OpenAI client.
The search method now embeds the query before retrieving the top matches.


In [94]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [95]:
vector_assistant.rag("How do I run Kafka??")

"I don't know."

In [96]:
"""
======================================================================
Closing the sqlitesearch index
======================================================================
- Close the connection when you're done using the index.
- This releases resources and finalizes the persistent index cleanly.
"""

vs_index.close()

print("Closed the sqlitesearch vector index connection.")

Closed the sqlitesearch vector index connection.
